In [ ]:
import sys
from pathlib import Path

def _find_project_root(start: Path) -> Path:
    for p in (start, *start.parents):
        if (p / ".git").exists():
            return p
    raise RuntimeError("Could not locate project root (no .git found above cwd)")

PROJECT_ROOT = _find_project_root(Path.cwd())
sys.path.insert(0, str(PROJECT_ROOT))
from src.seed_run import experiments_root, model_seed, seed_split_path  # noqa: E402
EXPERIMENTS_DIR = experiments_root(PROJECT_ROOT) / "mlp" / "experience_replay"
EXPERIMENTS_DIR.mkdir(parents=True, exist_ok=True)  # a seed tree starts empty; models_dir/CHECKPOINT_DIR below only mkdir one level


# MLP Experience Replay
Train an MLPClassifier with year-wise incremental scaling and experience replay.

In [ ]:
import copy
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from tqdm.notebook import tqdm

from src.mlp_replay.data import (
    load_dataset_and_splits,
    prepare_raw_features_for_year,
    prepare_features_for_year,
    precompute_yearly_raw_cache,
)
from src.mlp_replay.model import (
    build_mlp_model,
    capture_model_state,
    restore_model_state,
)
from src.mlp_replay.replay_strategies import (
    build_replay_year_spans,
    materialize_replay_samples_in_order,
)
from src.mlp_replay.training_loop import (
    resume_or_init_ratio_state,
    load_year_batch_or_none,
    finalize_ratio_and_save,
    write_combined_history_csv,
)
from src.mlp_replay.checkpointing import (
    create_empty_training_history,
    load_all_training_histories,
    save_all_training_histories,
    load_completion_status,
    save_completion_status,
    append_training_log,
    format_ratio_key,
    get_cached_raw_year,
)

warnings.filterwarnings('ignore')

## 1. Load Data and Splits

In [ ]:
ds_path = PROJECT_ROOT / 'training_data_with_features_plus_monthly_indices.zarr'
split_path = seed_split_path(PROJECT_ROOT)
ds, train_pixel_indices, val_pixel_indices, test_pixel_indices = load_dataset_and_splits(ds_path, split_path)


## 2. Feature Engineering

`prepare_raw_features_for_year` and `prepare_features_for_year` now live in `src/mlp_replay/data.py` and are imported above.


## 3. Initialize Class Weights and MLP

In [ ]:
print('Precomputing raw yearly features for train/validation splits...')
n_years = len(ds.year)


train_feature_cache = precompute_yearly_raw_cache(ds, train_pixel_indices, n_years, 'train', ds_path=ds_path)
val_feature_cache = precompute_yearly_raw_cache(ds, val_pixel_indices, n_years, 'validation', ds_path=ds_path)

print('Computing class weights from cached training labels...')
train_label_batches = [y_batch for _, y_batch in train_feature_cache.values() if len(y_batch) > 0]
if not train_label_batches:
    raise ValueError('No valid training labels after filtering.')

all_train_labels = np.concatenate(train_label_batches).astype(int)
all_train_labels = all_train_labels[np.isin(all_train_labels, [0, 1])]
if len(all_train_labels) == 0:
    raise ValueError('No valid training labels after filtering.')

classes = np.array([0, 1])
class_weights_array = compute_class_weight('balanced', classes=classes, y=all_train_labels)
class_weight_dict = {classes[i]: class_weights_array[i] for i in range(len(classes))}

print('Class weights:')
print(f"  Class 0: {class_weight_dict[0]:.4f}")
print(f"  Class 1: {class_weight_dict[1]:.4f}")

model = MLPClassifier(
    hidden_layer_sizes=(64,),
    activation='relu',
    alpha=0.0001,
    random_state=model_seed(),
    solver='adam',
    learning_rate='adaptive',
    max_iter=1,
    learning_rate_init=0.001,
    warm_start=False,
    verbose=False,
)
print('MLP initialized')

## 4. Replay-Enabled Online Training

In [ ]:
import json
from datetime import datetime

REPLAY_RATIOS = [0.3]
REPLAY_ENABLED = True
REPLAY_RANDOM_STATE = model_seed()
MISCLASS_FRACTION = 0.5

CHUNK_SIZE = 50000
MAX_EPOCHS = 15
PATIENCE = 3
MIN_DELTA = 0.0005

CHECKPOINT_DIR = EXPERIMENTS_DIR / 'training_checkpoints_mlp_missclassification_buffer'
ALL_HISTORIES_FILE = CHECKPOINT_DIR / 'mlp_missclassification_buffer_all_training_histories.pkl'
COMPLETION_STATUS_FILE = CHECKPOINT_DIR / 'mlp_missclassification_buffer_completion_status.json'
TRAINING_LOG_FILE = CHECKPOINT_DIR / 'mlp_missclassification_buffer_training_log.txt'
COMBINED_HISTORY_FILENAME = 'mlp_classifier_history_prevyears_monthly_features_incremental_scaler_missclassification_buffer_all_ratios.csv'
MISCLASS_SNAPSHOT_TEMPLATE = 'mlp_missclassification_buffer_misclass_snapshot_{ratio_key}_year_{year}.pkl'

CHECKPOINT_DIR.mkdir(exist_ok=True)

if 'train_feature_cache' not in globals() or 'val_feature_cache' not in globals():
    raise ValueError('Raw feature caches not found. Run Cell 8 first to precompute yearly features.')

def get_misclass_snapshot_path(checkpoint_dir, ratio_key, year_val):
    filename = MISCLASS_SNAPSHOT_TEMPLATE.format(ratio_key=ratio_key, year=year_val)
    return checkpoint_dir / filename

def save_misclass_snapshot(path, misclass_local_indices):
    serializable = {}
    for past_year_idx, idx_array in misclass_local_indices.items():
        serializable[int(past_year_idx)] = np.asarray(idx_array, dtype=np.int64)
    with open(path, 'wb') as f:
        pickle.dump(serializable, f)

def load_misclass_snapshot(path):
    if not path.exists():
        return {}
    with open(path, 'rb') as f:
        data = pickle.load(f)
    if not isinstance(data, dict):
        return {}

    cleaned = {}
    for key, value in data.items():
        cleaned[int(key)] = np.asarray(value, dtype=np.int64)
    return cleaned

def compute_misclassified_local_indices(model, scaler, train_cache, max_past_year_idx):
    misclass_local = {}
    for past_year_idx in range(1, max_past_year_idx + 1):
        X_past_raw, y_past = get_cached_raw_year(train_cache, past_year_idx)
        if len(X_past_raw) == 0:
            continue

        X_past_scaled = scaler.transform(X_past_raw)
        y_past_pred = model.predict(X_past_scaled)
        misclass_idx = np.flatnonzero(y_past_pred != y_past)
        misclass_local[past_year_idx] = misclass_idx.astype(np.int64, copy=False)

    return misclass_local

def build_replay_pools(train_cache, current_year_idx, label_dtype):
    """Span layout + labels for the replay pool. The FEATURES are deliberately
    not concatenated: this used to vstack every prior year (~3.6GB at the last
    training year, ~7.2GB with the duplicate alive beside it) purely so the
    sampled rows could be indexed out of it, which is what the Kaggle OOM
    killer was reacting to. materialize_replay_samples_in_order now fetches
    exactly the sampled rows, one year at a time, from the same cache.

    build_misclass_pool_indices and the sampling below are untouched -- they
    only ever worked on the global index space these spans define.
    """
    spans, y_replay_pool, _ = build_replay_year_spans(
        lambda past_year_idx: get_cached_raw_year(train_cache, past_year_idx)[1],
        current_year_idx,
    )
    if len(y_replay_pool) == 0:
        y_replay_pool = np.empty((0,), dtype=label_dtype)
    return y_replay_pool, spans

def build_misclass_pool_indices(misclass_local_indices, replay_year_spans):
    misclass_global = []
    for past_year_idx, start, end in replay_year_spans:
        n_samples = end - start
        local_idx = misclass_local_indices.get(past_year_idx)
        if local_idx is None or len(local_idx) == 0:
            continue

        valid_local = local_idx[(local_idx >= 0) & (local_idx < n_samples)]
        if len(valid_local) == 0:
            continue

        misclass_global.append(start + valid_local)

    if not misclass_global:
        return np.empty((0,), dtype=np.int64)

    return np.unique(np.concatenate(misclass_global)).astype(np.int64, copy=False)

all_training_histories = load_all_training_histories(ALL_HISTORIES_FILE)
completion_status = load_completion_status(COMPLETION_STATUS_FILE)

n_years = len(ds.year)
year_values = ds.year.values
year_value_to_idx = {int(y): idx for idx, y in enumerate(year_values)}
all_target_year_values = [int(year_values[idx]) for idx in range(1, n_years)]

print(f'Checkpoint directory: {CHECKPOINT_DIR.resolve()}')
print(f'Replay ratios: {REPLAY_RATIOS}')
append_training_log(TRAINING_LOG_FILE, f'Started training run for ratios: {REPLAY_RATIOS}')

for ratio_idx, replay_ratio in enumerate(REPLAY_RATIOS):
    ratio_key = format_ratio_key(replay_ratio)
    output_suffix = f'incremental_scaler_missclassification_buffer_{ratio_key}'
    models_dir_name = f'models_mlp_prevyears_monthly_features_{output_suffix}'
    scaler_file_template = f'scaler_year_{{year}}_mlp_prevyears_monthly_features_{output_suffix}.pkl'
    model_file_template = f'model_year_{{year}}_mlp_prevyears_monthly_features_{output_suffix}.pkl'
    final_scaler_filename = f'scaler_final_mlp_prevyears_monthly_features_{output_suffix}.pkl'
    final_model_filename = f'mlp_classifier_model_prevyears_monthly_features_{output_suffix}.pkl'
    history_filename = f'mlp_classifier_history_prevyears_monthly_features_{output_suffix}.csv'

    models_dir = EXPERIMENTS_DIR / models_dir_name
    models_dir.mkdir(exist_ok=True)

    if ratio_key in completion_status.get('completed_ratios', []):
        print(f'[{ratio_key}] already completed. Skipping ratio.')
        append_training_log(TRAINING_LOG_FILE, f'[{ratio_key}] skipped (already completed).')
        continue

    training_history = all_training_histories.get(ratio_key, create_empty_training_history(extra_keys=['misclass_pool_size', 'misclass_target_size', 'misclass_used_size', 'random_used_size']))
    for key in create_empty_training_history(extra_keys=['misclass_pool_size', 'misclass_target_size', 'misclass_used_size', 'random_used_size']).keys():
        training_history.setdefault(key, [])

    completed_years = set(int(y) for y in completion_status.get('completed_years', {}).get(ratio_key, []))
    completed_years.update(int(y) for y in training_history.get('year', []))
    completion_status.setdefault('completed_years', {})[ratio_key] = sorted(list(completed_years))

    replay_rng = np.random.default_rng(REPLAY_RANDOM_STATE + ratio_idx)

    model = None
    incremental_scaler = None
    start_year_idx = 1
    misclass_local_indices = {}

    if completed_years:
        resume_candidate_years = sorted(completed_years, reverse=True)
        resumed = False
        for resume_year in resume_candidate_years:
            year_model_path = models_dir / model_file_template.format(year=resume_year)
            year_scaler_path = models_dir / scaler_file_template.format(year=resume_year)
            if year_model_path.exists() and year_scaler_path.exists() and resume_year in year_value_to_idx:
                with open(year_model_path, 'rb') as f:
                    model = pickle.load(f)
                with open(year_scaler_path, 'rb') as f:
                    incremental_scaler = pickle.load(f)
                start_year_idx = year_value_to_idx[resume_year] + 1

                misclass_snapshot_path = get_misclass_snapshot_path(CHECKPOINT_DIR, ratio_key, resume_year)
                misclass_local_indices = load_misclass_snapshot(misclass_snapshot_path)

                resumed = True
                print(f'[{ratio_key}] resuming from year {resume_year}; continuing at index {start_year_idx}.')
                append_training_log(TRAINING_LOG_FILE, f'[{ratio_key}] resumed from year {resume_year}.')
                break

        if resumed:
            # Clean up completed_years and training_history for years after the resumed year
            completed_years = {y for y in completed_years if y <= resume_year}
            if training_history.get('year'):
                keep_indices = [i for i, y in enumerate(training_history['year']) if y <= resume_year]
                for key in training_history.keys():
                    if isinstance(training_history[key], list):
                        training_history[key] = [training_history[key][i] for i in keep_indices]
        else:
            print(f'[{ratio_key}] checkpoint artifacts missing/inconsistent. Restarting this ratio from scratch.')
            append_training_log(TRAINING_LOG_FILE, f'[{ratio_key}] restart due to missing/inconsistent checkpoint artifacts.')
            completed_years = set()
            completion_status['completed_years'][ratio_key] = []
            training_history = create_empty_training_history(extra_keys=['misclass_pool_size', 'misclass_target_size', 'misclass_used_size', 'random_used_size'])
            misclass_local_indices = {}

    if model is None:
        model = build_mlp_model()
    if incremental_scaler is None:
        incremental_scaler = StandardScaler()

    print(f'[{ratio_key}] Model output directory: {models_dir.resolve()}')
    print(f'[{ratio_key}] Years to train: 1..{n_years - 1} (year 0 skipped)')

    for year_idx in tqdm(range(start_year_idx, n_years), desc=f'{ratio_key} by year'):
        year_val = int(year_values[year_idx])

        if year_val in completed_years:
            continue

        year_batch = load_year_batch_or_none(
            train_feature_cache, val_feature_cache, year_idx, year_val, ratio_key,
            completed_years, completion_status, COMPLETION_STATUS_FILE, TRAINING_LOG_FILE,
        )
        if year_batch is None:
            continue
        X_train_raw, y_train_batch, X_val_raw, y_val_batch = year_batch

        incremental_scaler.partial_fit(X_train_raw)
        X_train_batch = incremental_scaler.transform(X_train_raw)
        X_val_batch = incremental_scaler.transform(X_val_raw)

        scaler_checkpoint = copy.deepcopy(incremental_scaler)
        year_scaler_path = models_dir / scaler_file_template.format(year=year_val)
        with open(year_scaler_path, 'wb') as f:
            pickle.dump(scaler_checkpoint, f)

        n_samples = len(X_train_batch)
        replay_target_size = int(n_samples * replay_ratio) if REPLAY_ENABLED else 0

        y_replay_pool, replay_year_spans = build_replay_pools(
            train_feature_cache,
            year_idx,
            y_train_batch.dtype,
        )

        def _load_raw_past_year(past_year_idx):
            return get_cached_raw_year(train_feature_cache, past_year_idx)

        replay_pool_size = len(y_replay_pool)
        replay_used_size = min(replay_target_size, replay_pool_size) if REPLAY_ENABLED else 0

        misclass_pool_indices = build_misclass_pool_indices(misclass_local_indices, replay_year_spans)
        misclass_pool_size = len(misclass_pool_indices)

        if hasattr(model, 'n_features_in_') and int(model.n_features_in_) != int(X_train_batch.shape[1]):
            raise ValueError(
                f'Feature count mismatch at year {year_val}: model expects {model.n_features_in_}, got {X_train_batch.shape[1]}'
            )

        # One-time yearly sampling for replay segments (no resampling per epoch).
        misclass_target_size = 0
        misclass_used_size = 0
        random_used_size = 0

        if replay_used_size > 0:
            epoch_replay_target = min(replay_target_size, replay_pool_size)

            # Target is 50% misclassified and 50% random when enough samples exist.
            misclass_target_size = int(np.floor(epoch_replay_target * MISCLASS_FRACTION))
            random_target_size = epoch_replay_target - misclass_target_size

            sampled_indices_parts = []
            selected_misclass = np.empty((0,), dtype=np.int64)

            if misclass_pool_size > 0 and misclass_target_size > 0:
                misclass_take = min(misclass_target_size, misclass_pool_size)
                misclass_choice_pos = replay_rng.choice(misclass_pool_size, size=misclass_take, replace=False)
                selected_misclass = misclass_pool_indices[misclass_choice_pos]
                sampled_indices_parts.append(selected_misclass)
            else:
                misclass_take = 0

            remaining_needed = random_target_size + (misclass_target_size - misclass_take)

            selected_random = np.empty((0,), dtype=np.int64)
            if remaining_needed > 0 and replay_pool_size > 0:
                full_pool = np.arange(replay_pool_size, dtype=np.int64)
                if len(selected_misclass) > 0:
                    available_random = np.setdiff1d(full_pool, selected_misclass, assume_unique=False)
                else:
                    available_random = full_pool

                if len(available_random) > 0:
                    random_take = min(remaining_needed, len(available_random))
                    random_choice_pos = replay_rng.choice(len(available_random), size=random_take, replace=False)
                    selected_random = available_random[random_choice_pos]
                    sampled_indices_parts.append(selected_random)

            if sampled_indices_parts:
                replay_indices = np.concatenate(sampled_indices_parts)
                replay_used_size = len(replay_indices)
                misclass_used_size = len(selected_misclass)
                random_used_size = len(selected_random)

                X_replay_sampled, y_replay_sampled = materialize_replay_samples_in_order(
                    replay_indices, replay_year_spans, _load_raw_past_year, incremental_scaler,
                )
                X_combined_base = np.concatenate([X_train_batch, X_replay_sampled], axis=0)
                y_combined_base = np.concatenate([y_train_batch, y_replay_sampled], axis=0)
            else:
                replay_used_size = 0
                misclass_used_size = 0
                random_used_size = 0
                X_combined_base = X_train_batch
                y_combined_base = y_train_batch
        else:
            misclass_target_size = 0
            misclass_used_size = 0
            random_used_size = 0
            X_combined_base = X_train_batch
            y_combined_base = y_train_batch

        best_val_pr_auc = -np.inf
        patience_counter = 0
        best_model_state = None

        for epoch in range(MAX_EPOCHS):
            combined_n_samples = len(X_combined_base)
            shuffle_idx = replay_rng.permutation(combined_n_samples)
            X_train_shuffled = X_combined_base[shuffle_idx]
            y_train_shuffled = y_combined_base[shuffle_idx]
            n_chunks = max(1, int(np.ceil(combined_n_samples / CHUNK_SIZE)))

            for chunk_idx in range(n_chunks):
                start_idx = chunk_idx * CHUNK_SIZE
                end_idx = min(start_idx + CHUNK_SIZE, combined_n_samples)
                X_chunk = X_train_shuffled[start_idx:end_idx]
                y_chunk = y_train_shuffled[start_idx:end_idx]
                sample_weights_chunk = np.array([class_weight_dict[int(label)] for label in y_chunk])
                model.partial_fit(X_chunk, y_chunk, classes=classes, sample_weight=sample_weights_chunk)

            y_val_pred = model.predict(X_val_batch)
            y_val_proba = model.predict_proba(X_val_batch)[:, 1]
            val_pr_auc = average_precision_score(y_val_batch, y_val_proba) if len(np.unique(y_val_batch)) > 1 else np.nan

            if val_pr_auc > best_val_pr_auc + MIN_DELTA:
                best_val_pr_auc = val_pr_auc
                patience_counter = 0
                best_model_state = capture_model_state(model)
            else:
                patience_counter += 1
                if patience_counter >= PATIENCE:
                    break

        if best_model_state is not None:
            restore_model_state(model, best_model_state)

        y_train_pred = model.predict(X_train_batch)
        y_val_pred = model.predict(X_val_batch)
        y_val_proba = model.predict_proba(X_val_batch)[:, 1]

        train_acc = accuracy_score(y_train_batch, y_train_pred)
        train_prec = precision_score(y_train_batch, y_train_pred, zero_division=0)
        train_rec = recall_score(y_train_batch, y_train_pred, zero_division=0)
        train_f1 = f1_score(y_train_batch, y_train_pred, zero_division=0)

        val_acc = accuracy_score(y_val_batch, y_val_pred)
        val_prec = precision_score(y_val_batch, y_val_pred, zero_division=0)
        val_rec = recall_score(y_val_batch, y_val_pred, zero_division=0)
        val_f1 = f1_score(y_val_batch, y_val_pred, zero_division=0)
        if len(np.unique(y_val_batch)) > 1:
            val_roc_auc = roc_auc_score(y_val_batch, y_val_proba)
            val_pr_auc = average_precision_score(y_val_batch, y_val_proba)
        else:
            val_roc_auc = np.nan
            val_pr_auc = np.nan

        training_history['year'].append(year_val)
        training_history['train_accuracy'].append(train_acc)
        training_history['train_precision'].append(train_prec)
        training_history['train_recall'].append(train_rec)
        training_history['train_f1'].append(train_f1)
        training_history['val_accuracy'].append(val_acc)
        training_history['val_precision'].append(val_prec)
        training_history['val_recall'].append(val_rec)
        training_history['val_f1'].append(val_f1)
        training_history['val_roc_auc'].append(val_roc_auc)
        training_history['val_pr_auc'].append(val_pr_auc)
        training_history['replay_pool_size'].append(int(replay_pool_size))
        training_history['replay_target_size'].append(int(replay_target_size))
        training_history['replay_used_size'].append(int(replay_used_size))
        training_history['misclass_pool_size'].append(int(misclass_pool_size))
        training_history['misclass_target_size'].append(int(misclass_target_size))
        training_history['misclass_used_size'].append(int(misclass_used_size))
        training_history['random_used_size'].append(int(random_used_size))

        if REPLAY_ENABLED and year_idx >= 1:
            misclass_local_indices = compute_misclassified_local_indices(
                model,
                incremental_scaler,
                train_feature_cache,
                year_idx,
            )
            misclass_snapshot_path = get_misclass_snapshot_path(CHECKPOINT_DIR, ratio_key, year_val)
            save_misclass_snapshot(misclass_snapshot_path, misclass_local_indices)

        year_model_path = models_dir / model_file_template.format(year=year_val)
        with open(year_model_path, 'wb') as f:
            pickle.dump(model, f)

        completed_years.add(year_val)
        completion_status['completed_years'][ratio_key] = sorted(list(completed_years))
        all_training_histories[ratio_key] = training_history.copy()
        save_all_training_histories(ALL_HISTORIES_FILE, all_training_histories)
        save_completion_status(COMPLETION_STATUS_FILE, completion_status)

        print(
            f'[{ratio_key}] Year {year_val}: Train F1={train_f1:.3f}, Val F1={val_f1:.3f}, Val PR-AUC={val_pr_auc:.3f}, '
            f'replay_used={replay_used_size:,}/{replay_pool_size:,}, misclass_used={misclass_used_size:,}, random_used={random_used_size:,}'
        )
        append_training_log(
            TRAINING_LOG_FILE,
            f'[{ratio_key}] completed year {year_val} with replay_used={replay_used_size}/{replay_pool_size}, '
            f'misclass_used={misclass_used_size}, random_used={random_used_size}.',
        )

    finalize_ratio_and_save(
        incremental_scaler, model, models_dir, final_scaler_filename, final_model_filename,
        history_filename, training_history, ratio_key, all_target_year_values, completed_years,
        completion_status, COMPLETION_STATUS_FILE, TRAINING_LOG_FILE, all_training_histories, ALL_HISTORIES_FILE,
    )

write_combined_history_csv(all_training_histories, EXPERIMENTS_DIR / COMBINED_HISTORY_FILENAME)

save_completion_status(COMPLETION_STATUS_FILE, completion_status)
save_all_training_histories(ALL_HISTORIES_FILE, all_training_histories)
append_training_log(TRAINING_LOG_FILE, 'Training run completed.')

## 5. Save Training History

In [6]:
import json

CHECKPOINT_DIR = EXPERIMENTS_DIR / 'training_checkpoints_mlp_missclassification_buffer'
ALL_HISTORIES_FILE = CHECKPOINT_DIR / 'mlp_missclassification_buffer_all_training_histories.pkl'
COMPLETION_STATUS_FILE = CHECKPOINT_DIR / 'mlp_missclassification_buffer_completion_status.json'
COMBINED_HISTORY_FILENAME = 'mlp_classifier_history_prevyears_monthly_features_incremental_scaler_missclassification_buffer_all_ratios.csv'

if ALL_HISTORIES_FILE.exists():
    with open(ALL_HISTORIES_FILE, 'rb') as f:
        all_training_histories = pickle.load(f)
else:
    all_training_histories = {}

if COMPLETION_STATUS_FILE.exists():
    with open(COMPLETION_STATUS_FILE, 'r', encoding='utf-8') as f:
        completion_status = json.load(f)
else:
    completion_status = {'completed_ratios': [], 'completed_years': {}}

summary_rows = []
for ratio_key in sorted(all_training_histories.keys()):
    history_dict = all_training_histories[ratio_key]
    n_rows = len(history_dict.get('year', []))
    last_year = history_dict['year'][-1] if n_rows > 0 else np.nan
    last_val_f1 = history_dict['val_f1'][-1] if n_rows > 0 else np.nan
    last_val_pr_auc = history_dict['val_pr_auc'][-1] if n_rows > 0 else np.nan
    is_completed = ratio_key in completion_status.get('completed_ratios', [])
    summary_rows.append(
        {
            'ratio_key': ratio_key,
            'rows': n_rows,
            'last_year': last_year,
            'last_val_f1': last_val_f1,
            'last_val_pr_auc': last_val_pr_auc,
            'completed': is_completed,
        }
    )

summary_df = pd.DataFrame(summary_rows).sort_values('ratio_key').reset_index(drop=True)
print('Per-ratio training summary:')
display(summary_df)

combined_history_path = EXPERIMENTS_DIR / COMBINED_HISTORY_FILENAME
if combined_history_path.exists():
    combined_df = pd.read_csv(combined_history_path)
    print(f'Combined history file: {combined_history_path}')
    print(f'Rows: {len(combined_df):,}')
    display(combined_df.tail())
else:
    print(f'Combined history file not found: {combined_history_path}')

Per-ratio training summary:


,ratio_key,rows,last_year,last_val_f1,last_val_pr_auc,completed
0,RR_0.3,6,2022,0.194536,0.361646,True


Combined history file: mlp_classifier_history_prevyears_monthly_features_incremental_scaler_missclassification_buffer_all_ratios.csv
Rows: 6


,year,train_accuracy,train_precision,train_recall,train_f1,val_accuracy,val_precision,val_recall,val_f1,val_roc_auc,val_pr_auc,replay_pool_size,replay_target_size,replay_used_size,misclass_pool_size,misclass_target_size,misclass_used_size,random_used_size,ratio_key,replay_ratio
1,2018,0.893187,0.136184,0.786072,0.232148,0.889114,0.135356,0.744041,0.229044,0.903228,0.436349,5575648,1676042,1676042,910072,838021,838021,838021,RR_0.3,0.3
2,2019,0.879167,0.119856,0.782911,0.207886,0.886553,0.113473,0.764230,0.197606,0.905855,0.430290,11162455,1676031,1676031,2940415,838015,838015,838016,RR_0.3,0.3
3,2020,0.866709,0.104420,0.812384,0.185054,0.869611,0.092639,0.762620,0.165210,0.901446,0.456488,16749226,1676217,1676217,3701004,838108,838108,838109,RR_0.3,0.3
4,2021,0.917533,0.129702,0.773677,0.222161,0.908138,0.115001,0.731657,0.198761,0.899998,0.417045,22336618,1676056,1676056,3912397,838028,838028,838028,RR_0.3,0.3
5,2022,0.810009,0.125202,0.868365,0.218850,0.809670,0.110783,0.797307,0.194536,0.886414,0.361646,27923473,1677624,1677624,5776841,838812,838812,838812,RR_0.3,0.3
